# 不使用@tool的方式定义工具

## 1、举例：

In [ ]:
# 1、模型初始化
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL")
)

# 2、声明一个函数(工具)
def get_weather(city : str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("北京的天气怎么样？")
rprint(response)

## 2、工具描述的各部分详解

执行model.bind_tools([get_weather])，底层最终会调用convert_to_openai_tool生成工具描述。所
以我们可以直接调用后者查看解析后的工具描述

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city:str):
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

## 2.2 description说明

举例：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city : str):
    """
     查询城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

## 2.3 参数说明

举例：

In [ ]:
def get_weather(city : str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

## 2.4 参数类型的说明

举例1：

In [ ]:
def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

## 2.4 参数类型的说明

举例2：

In [ ]:
def get_weather(city):
    """
    查询城市的天气

    Args:
        city:具体的城市
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

## 2.5 参数默认值的说明

一旦参数设置了默认值，则打印结果中的required字段中就不再包含此参数

举例1：

In [16]:
def get_weather(city : str = "北京"):
    """
    查询城市的天气

    Args:
        city:具体的城市
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [17]:
def get_weather(dt : str ,city : str = "北京"):
    """
    查询城市的天气

    Args:
        city:具体的城市
        dt:时间
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}